# CASE 1 — MNIST, IID, JANUS-FL

Runs the full round protocol (ARCHITECTURE.md §4) over MNIST with the
stratified-IID split (fl/splits.py), for both MVP backbones
(ResNet-50, MobileNetV2), ~45 rounds, 1 seed — see
`configs/mvp_3day.yaml`. **Run `00_Setup_and_Crypto_Tests.ipynb`
first.**


In [ ]:
import sys, yaml
sys.path.insert(0, '/content/FL-ZKP')
%cd /content/FL-ZKP

with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


## Load data and compute the stratified-IID split

In [ ]:
from fl.data_iid import prepare_case1

mnist_cfg = CFG['datasets']['mnist']
case1_data = prepare_case1(seed=CFG['seed'], source=mnist_cfg['source'], csv_path=mnist_cfg.get('csv_path'))
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


## Build the ring (one keypair per client) and the trace committee

In [ ]:
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

# Separate X25519 keypairs for secure-aggregation's pairwise masking --
# a DIFFERENT curve from the ristretto255 signing keys above (see
# crypto/secagg.py's own note). Long-term, computed once here.
x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


## Build models (2 backbones, from scratch, GroupNorm)

In [ ]:
from fl.models import build_model, flatten_state_dict

models = {name: build_model(name, num_classes=10, in_channels=1) for name in CFG['mvp_scope']['models']}
flat_weights = {name: flatten_state_dict(m)[0] for name, m in models.items()}
shapes = {name: flatten_state_dict(m)[1] for name, m in models.items()}
for name, w in flat_weights.items():
    print(name, "param count:", w.numel())


## Round loop (per model) -- orchestrates fl/client.py + fl/server.py over configs/mvp_3day.yaml's rounds_per_run

**Read this before running the next cell.** `crypto/sketch.py`'s
`SketchProjector` is a CPU/NumPy CORRECTNESS reference, not a
performance one (see its own PERFORMANCE NOTE): measured at ~16
min/client/round extrapolated to ResNet-50's real d~23.5M. The full
45-round x 50-client x 2-model run would take DAYS with this
reference implementation. **Start with the smoke test below** (a
handful of rounds/clients) to validate the wiring, not the full
`CFG['mvp_scope']['rounds_per_run']` sweep -- see STATUS.md TODO #3
for the GPU/torch reimplementation this blocks on.


In [ ]:
import numpy as np
from fl.round_loop import run_training_loop

# SMOKE TEST first -- a handful of rounds over a SUBSET of clients,
# to validate the protocol wiring (V1-V5, the independent V5 audit,
# B_t evolution) runs end to end before committing to the full,
# much slower, MVP-scale sweep. Bump these up once this passes clean.
SMOKE_ROUNDS = 2
SMOKE_CLIENTS = 8  # must be >= K_MIN (8) in fl/server.py or every round aborts regardless of verification outcome

results = {}
for model_name in CFG['mvp_scope']['models']:
    print(f"\n==== {model_name} (smoke test: {SMOKE_ROUNDS} rounds, {SMOKE_CLIENTS} clients) ====")
    model = models[model_name]
    dim = flat_weights[model_name].numel()

    # round-0 bootstrap for B_t from server_root. NOTE: still a
    # placeholder (see fl/round_loop.py's own docstring) -- real norms
    # would come from the server running its own pseudo-client
    # train+sketch passes over server_root data, which hits the same
    # sketch performance wall noted above, so it's deferred alongside it.
    root_norms = np.random.default_rng(CFG['seed']).normal(5.0, 0.5, size=100)

    final_w_t, history = run_training_loop(
        model=model,
        model_dim=dim,
        client_datasets=case1_data['client_train'][:SMOKE_CLIENTS],
        ring=ring[:SMOKE_CLIENTS],
        client_sks=client_sks[:SMOKE_CLIENTS],
        pairwise_secrets={i: {j: pairwise_secrets[i][j] for j in range(SMOKE_CLIENTS) if j != i} for i in range(SMOKE_CLIENTS)},
        pk_trace=PK_trace,
        server_root_norms=root_norms,
        CFG=CFG,
        rounds=SMOKE_ROUNDS,
    )
    results[model_name] = {'final_w_t': final_w_t, 'history': history}


## Known gaps, flagged honestly

Resolved in this version (was a scaffold that stopped before any real
per-client wiring; now a complete `fl/round_loop.py::run_training_loop`
covering the full VRF-sample -> train -> mask -> commit -> escrow ->
sign -> sketch -> range-proof -> R_cons -> V1-V5 -> independent V5
audit -> apply_update -> B_t-EMA-update sequence, shared between CASE1
and CASE2 so they can't drift apart):

- Real per-client pairwise ECDH secrets (`fl.round_loop.build_x25519_keys`
  + `precompute_pairwise_secrets`), not an empty placeholder dict.
- Real local data threaded through (`case1_data['client_train'][client_idx]`).
- The sketch-seed reveal ordering (`sigma_t` picked and committed to
  via `seed_commitment` BEFORE phase 1, "revealed" for phase 2 after).
- V1-V5 verification, the INDEPENDENT V5 re-check
  (`verifier/audit.py::independent_v5_check`, deliberately a separate
  implementation from `fl/server.py`'s own computation), and
  `apply_update`.
- `B_t` evolving round to round via `fl/norm_rule.py`'s EMA update,
  not recomputed from scratch each time.

Still open, not silently hidden:

1. **The sketch performance wall above.** Run the smoke test first;
   do not launch the full MVP-scale sweep before
   `crypto/sketch.py`'s GPU/torch reimplementation (STATUS.md TODO #3).
2. **`server_root_norms` is still a placeholder** (see the round-loop
   cell's own comment) -- affects only round 0's starting `B_t`; every
   later round's `B_t` comes from real submissions' real sketch norms.
3. **Evidence accumulation / calibration (`fl/evidence.py`) is not
   wired into the round loop yet** -- it needs per-client loss deltas
   that `fl/client.py::local_train_delta` doesn't currently return, and
   a per-round median-reference vector across all submissions. This is
   its own piece of wiring, separate from the V1-V5/aggregation loop
   above, and is tracked as a follow-up, not assumed done.
